In [ ]:
import numpy as np
from scipy.io import loadmat

# File path, type pair, and selected measurement windows (stop is exclusive)
file_path = 'data_simulation/structure_function'
type_pair = "A_A"  # Options: "A_A", "A_B", "B_B"
iterations = range(100000, 500000, 100000)
file_names = [f"{file_path}_{step:06d}.mat" for step in iterations]
if not file_names:
    raise ValueError("Select at least one structure-function file.")

# Load the grid and cell from the first selected file
geometry = loadmat(file_names[0], squeeze_me=True, variable_names=['nx', 'lx'])
nx = geometry['nx'].astype(int)
lx = geometry['lx']

# Weight file means by the actual number of samples in each measurement window
sample_counts = []
for file_name in file_names:
    record = loadmat(file_name, variable_names=['sample_count'])
    if 'sample_count' not in record:
        sample_counts.append(None)
        continue
    count = np.asarray(record['sample_count'])
    if (count.size != 1 or count.dtype.kind not in 'iuf' or not np.isfinite(count).all()
            or count.item() < 0 or count.item() != np.floor(count.item())):
        raise ValueError(f"sample_count must be a finite nonnegative integer in {file_name}.")
    sample_counts.append(float(count.item()))
# Legacy files all receive equal weight; unknown counts cannot be mixed with known counts.
if all(count is None for count in sample_counts):
    sample_counts = [1.0] * len(file_names)
elif any(count is None for count in sample_counts):
    raise ValueError("Cannot mix structure-function files with and without sample_count.")
if not any(count > 0 for count in sample_counts):
    raise ValueError("The selected structure-function files contain no samples.")

# Multiplicity of each kz plane in the full Fourier spectrum
mode_weights = np.full(nx[2]//2 + 1, 2.0)
mode_weights[0] = 1.0
if nx[2] % 2 == 0:
    mode_weights[-1] = 1.0

# Calculate k² and group Fourier modes with equal magnitude
k2 = np.zeros((nx[0], nx[1], nx[2]//2 + 1), dtype=np.float64)
for i in range(nx[0]):
    for j in range(nx[1]):
        for k in range(nx[2]//2 + 1):
            temp_i = min(i, nx[0] - i)
            temp_j = min(j, nx[1] - j)
            val = round((temp_i/lx[0])**2 + (temp_j/lx[1])**2 + (k/lx[2])**2, 7)
            k2[i, j, k] = val
k2_unique, inverse_indices = np.unique(k2, return_inverse=True)
k2_mapping = inverse_indices.reshape(k2.shape)

sf_mag = np.zeros_like(k2_unique)
sf_count = np.zeros_like(k2_unique)
for file_name, sample_count in zip(file_names, sample_counts):
    if sample_count == 0:
        continue
    print(file_name)
    data = loadmat(file_name, variable_names=[f'structure_function_{type_pair}'])
    v = data[f'structure_function_{type_pair}']
    for i in range(nx[0]):
        for j in range(nx[1]):
            for k in range(nx[2]//2 + 1):
                idx = k2_mapping[i, j, k]
                weight = mode_weights[k] * sample_count
                sf_mag[idx] += v[i, j, k].real * weight
                sf_count[idx] += weight

x = np.sqrt(k2_unique) * 2 * np.pi
y = sf_mag / sf_count
y[0] = 0.0  # Remove the k=0 mode


In [ ]:
import matplotlib.pyplot as plt

# Plot
plt.figure(dpi=100)
plt.style.use('classic')

plt.plot(x, y)
plt.title('Structure function')
plt.xlabel('$kR_0$')
plt.ylabel('$S(k)ρ_0N$')
plt.yscale('log')
plt.xlim([2, 14])
plt.ylim([0.01, 1000])
plt.grid(True)
plt.show()

In [ ]:
import pandas

# Write
df = pandas.DataFrame({'k': x, 's(k)': y})
df.to_excel("data.xlsx", sheet_name='sheet1', index=False)